# Digital Finance Word Cloud
  Stage 1: Year-by-year word clouds  
  Stage 2: Portrait word cloud  
  Stage 3: Read all images and make a video   

In [1]:
import os
import re
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageDraw, ImageFont
from wordcloud import WordCloud, STOPWORDS

In [2]:
# ============ Setting ============
CSV_FILE     = "Data/digital_finance_aftercleaning.csv"
ABSTRACT_COL = "abstract"
YEAR_COL     = "year"

# ---- Stage 1：year-by-year WordCloud ----
FONT_PATH = "C:/Windows/Fonts/arialbd.ttf"
MASK_W, MASK_H = 1600, 800          # WordCloud Size

# ---- Stage 2：portrait WordCloud ----
PORTRAIT     = "Data/mask.png"
THRESH       = 180
CLOSE_KERNEL = 25

# ---- Stage 3：video ----
WORDCLOUD_DIR = "Result/WordCloud"                        
VIDEO_FILE    = "Result/WordCloud/digital_finance_WordCloud.mp4"
PORTRAIT_NAME = "portrait.png"      # portrait
PORTRAIT_POSITION = "last"          # portrait position
HOLD_SEC = 2.0                      
FADE_SEC = 1.0
FPS = 30

In [3]:
# ============ Data and Stopwords ============
df = pd.read_csv(CSV_FILE)
df = df.dropna(subset=[ABSTRACT_COL, YEAR_COL])
df[YEAR_COL] = df[YEAR_COL].astype(int)

extra_stops = {
    "paper", "study", "using", "based", "article", "result", "results",
    "also", "however", "may", "use", "used", "shows", "show", "well",
    "within", "among", "toward", "towards", "findings", "finding",
    "suggest", "suggests", "provide", "provides", "important",
    "model", "models", "data", "market", "financial", "price",
    "methods", "prices", "approach",
    "role","different","overall","two","time","high","research","series","across","analysis","effect","framework"
}
STOPWORDS_ALL = STOPWORDS | extra_stops

text_by_year = {   # year-by-year text
    y: " ".join(df.loc[df[YEAR_COL] == y, ABSTRACT_COL].astype(str)).lower()
    for y in sorted(df[YEAR_COL].unique())
}
text_all = " ".join(df[ABSTRACT_COL].astype(str)).lower()   # 阶段 2 用：全量文本

In [4]:
# =====================================================================
# Stage 1：year-by-year WordCloud
# =====================================================================
def make_year_mask(year: int) -> np.ndarray:
    img = Image.new("L", (MASK_W, MASK_H), 255)
    draw = ImageDraw.Draw(img)
    text = str(year)

    lo, hi, best = 10, MASK_H, 10
    while lo <= hi:
        mid = (lo + hi) // 2
        font = ImageFont.truetype(FONT_PATH, mid)
        bbox = draw.textbbox((0, 0), text, font=font)
        if bbox[2] - bbox[0] <= MASK_W * 0.9:
            best, lo = mid, mid + 1
        else:
            hi = mid - 1

    font = ImageFont.truetype(FONT_PATH, best)
    bbox = draw.textbbox((0, 0), text, font=font)
    x = (MASK_W - (bbox[2] - bbox[0])) / 2 - bbox[0]
    y = (MASK_H - (bbox[3] - bbox[1])) / 2 - bbox[1]
    draw.text((x, y), text, font=font, fill=0)
    return np.array(img)

print("=" * 20, "Stage 1：year-by-year WordCloud", "=" * 20)
for y, text in text_by_year.items():
    out = f"{WORDCLOUD_DIR}/{y}.png"
    wc = WordCloud(
        width=MASK_W, height=MASK_H,
        background_color="white",
        mask=make_year_mask(y),
        stopwords=STOPWORDS_ALL,
        collocations=True,
        max_words=300,
        min_font_size=6,
        relative_scaling=0.4,
        random_state=42,
    ).generate(text)
    wc.to_image().save(out)
    print(f"  saved {out}")

==================== Stage 1：year-by-year WordCloud ====================
  saved Result/wordclouds/2019.png
  saved Result/wordclouds/2020.png
  saved Result/wordclouds/2021.png
  saved Result/wordclouds/2022.png
  saved Result/wordclouds/2023.png
  saved Result/wordclouds/2024.png
  saved Result/wordclouds/2025.png
  saved Result/wordclouds/2026.png


In [5]:
# =====================================================================
# Stage 2：portrait WordCloud
# =====================================================================
def outline_to_solid_mask(png_path: str) -> np.ndarray:
    img = Image.open(png_path)
    if img.mode in ("RGBA", "LA", "P"):
        img = img.convert("RGBA")
        img = Image.alpha_composite(
            Image.new("RGBA", img.size, (255, 255, 255, 255)), img)
    img = img.convert("L")

    ratio = MASK_W / img.width
    img = img.resize((MASK_W, int(img.height * ratio)), Image.LANCZOS)
    arr = np.array(img)

    bin_ = np.where(arr < THRESH, 0, 255).astype(np.uint8)
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (CLOSE_KERNEL, CLOSE_KERNEL))
    bin_ = cv2.morphologyEx(bin_, cv2.MORPH_CLOSE, kernel)

    h, w = bin_.shape
    ff = bin_.copy()
    pad = np.zeros((h + 2, w + 2), np.uint8)
    for seed in [(0, 0), (w - 1, 0), (0, h - 1), (w - 1, h - 1)]:
        if ff[seed[1], seed[0]] == 255:
            cv2.floodFill(ff, pad, seed, 128)

    interior = np.where(ff == 0, 0, 255).astype(np.uint8)
    solid = np.where(bin_ == 0, 0, interior)

    bin2 = (solid < 28).astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(bin2)
    if n > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        solid = np.where(labels == largest, 0, 255).astype(np.uint8)
    return solid


print("=" * 20, "Stage 2：portrait WordCloud", "=" * 20)
out = f"{WORDCLOUD_DIR}/{PORTRAIT_NAME}"
mask = outline_to_solid_mask(PORTRAIT)


mh, mw = mask.shape
wc = WordCloud(
    width=mw, height=mh,
    background_color="white",
    mask=mask,
    stopwords=STOPWORDS_ALL,
    collocations=True,
    max_words=400,
    min_font_size=5,
    relative_scaling=0.4,
    prefer_horizontal=0.9,
    colormap="viridis",
    contour_width=2,
    contour_color="darkred",
    random_state=42,
).generate(text_all)
wc.to_image().save(out)
print(f"  saved {out}")

==================== Stage 2：portrait WordCloud ====================
  saved Result/wordclouds/portrait.png


In [6]:
# =====================================================================
# Stage 3: Videos
# =====================================================================
def collect_frames() -> list:
    
    year_files = []
    for f in os.listdir(WORDCLOUD_DIR):
        if re.fullmatch(r"\d{4}\.png", f):          # year by year PNG
            year_files.append(f)
    year_files.sort()
    ordered = year_files[:]
    if os.path.exists(f"{WORDCLOUD_DIR}/{PORTRAIT_NAME}"):
        ordered.append(PORTRAIT_NAME) if PORTRAIT_POSITION == "last" \
            else ordered.insert(0, PORTRAIT_NAME)


    frames = []
    for f in ordered:
        img = Image.open(f"{WORDCLOUD_DIR}/{f}").convert("RGB")
        if img.size != (MASK_W, MASK_H):           
            img = img.resize((MASK_W, MASK_H), Image.LANCZOS)
        frames.append((f, np.array(img)))
    print(f"  Video frame sequence: {[f for f, _ in frames]}")   
    return frames


print("=" * 20, "Stage 3：make a video", "=" * 20)
frames = collect_frames()
writer = cv2.VideoWriter(
    VIDEO_FILE,
    cv2.VideoWriter_fourcc(*"mp4v"),   
    FPS, (MASK_W, MASK_H),
)
hold_n, fade_n = int(HOLD_SEC * FPS), int(FADE_SEC * FPS)

frames = [(f, cv2.cvtColor(img, cv2.COLOR_RGB2BGR)) for f, img in frames]

for i, (_, cur) in enumerate(frames):
    cur = cur.astype(np.float64)
    for _ in range(hold_n):                       
        writer.write(cur.astype(np.uint8))
    if i + 1 < len(frames):                       
        nxt = frames[i + 1][1].astype(np.float64)
        for k in range(1, fade_n + 1):
            alpha = k / fade_n
            writer.write((cur * (1 - alpha) + nxt * alpha).astype(np.uint8))

writer.release()
print(f"  saved: {VIDEO_FILE}")

==================== Stage 3：make a video ====================
  Video frame sequence: ['2019.png', '2020.png', '2021.png', '2022.png', '2023.png', '2024.png', '2025.png', '2026.png', 'portrait.png']
  saved: Result/wordclouds/digital_finance_WordCloud.mp4
